# Generating synthetic data for customers and items

The database of 40 customers and 50 products were generated using using Gemini 2.5 pro, which was prompted to generate the following information:

**Products**
- **PID**: int
- **Name**: str
- **Description**: str
- **Price**: float
- **Cost**: float
- **FeatureVector**: D-dimensional vector

**Customers**
- **CID**: int
- **Name**: str
- **Age**: int
- **Gender**: int
- **JobType**: str
- **Income**: int
- **FamilyStatus**: str
- **Segment**: str
- **FeatureVector**: D-dimensional vector

To generate the feature vectors, Gemini was prompted to consider the following D=5 features in accordance with the product description or customer information:
1. **Style**: Reflects product design, from exclusive/luxury (1.0) to generic/budget (−1.0)
2. **Functionality**: How useful or practical the product is, from highly functional (1.0) to purely decorative (−1.0)
3. **Durability**: How long-lasting or well-made the product is, from very durable (1.0) to disposable (−1.0)
4. **Sustainability**: How eco-friendly a product is, from sustainable (1.0) to standard (−1.0)
5. **Innovation**: How new or cutting-edge a product is, from highly innovative (1.0) to classic/traditional (−1.0)

A vector indicating the degree to which these 5 features apply is assigned to each customer and product, which enables affinity computation. In addition, customers are divided into one of the following segments based on purchase history:
1. Newcomers (N): Those who signed up but have made no purchases
2. At-risk Customers (A): Those who have made a purchase but last purchase was more than 90 days ago regardless of their average order value (AOV)
3. High-value customers (H): Those with a high AOV
4. Low-value customers (L): Those with a low AOV
5. Regular customers (R): Those with a modest AOV

## TODO: Generate transaction history data
Transaction history data would depend on the marketing history as well, but we decouple these two tables by passing into the transaction generator the parameter of Bernoulli measuring the quality of the marketing emails. The schema is as follows:

**Transactions**
- **TID**: int
- **CID**: int
- **Date**: date
- **Order**: List[(PID,quantity)] 
- **Designated delivery time window**: (timestamp,timestamp), ignored for now for simplicity

The logic of generating the history is as follows:
1. Disregard all the newcomers as they have no purchase history.
2. For the other customer segments, the shopping occurrences are modelled as Poisson point processes.
Each customer has a parameter based on their segment that determines the (exponential distributed, independent) arrival times. We sample the (continuous) time duration backwards to meet the date constraints and then discretize back into days (merging duplicate days).
The constraints for `A` is easily met by choosing the most recent time to be sufficiently long ago. The AOV constraints will be met by making minimal modifications to the generated data.
  

Parameters to keep track of:
- `days_since_last_purchase`: matrix of size $C\times I$, with values in $\{0,1,\ldots\}\cup\{\infty\}$
- `num_purchases`: int array of size $C$ counting the total number of orders placed by customer.
- `val_purchases`: float array of size $C$ indicating the total value of all past orders by customers.



In [2]:
# !pip3 install numpy
# !pip3 install pandas
# !pip3 install matplotlib
# !pip3 install peewee

In [1]:
import numpy as np
import pandas as pd
import datetime as dt
from collections import Counter
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

In [2]:
# System parameters
K = 10 # number of recommendations to display
C = 40 # number of customers
I = 50 # number of products
RHO = 0.99 # controls probability increase since last purchase
EPS = 1e-7 # small constant to avoid division by zero
AT_RISK_DAYS = 60  # min days since last purchase to be considered at-risk regular
AVG_VAL_H = 45 # average value threshold for high-value customers
AVG_VAL_L = 15 # average  value threshold for low-value customers
start_date = dt.date(2024, 10, 1) # start date for transactions
end_date = dt.date(2025, 10, 1) # end date for transactions
T = (end_date - start_date).days # total days in simulation

In [4]:
# Load data
products = pd.read_csv('products.tsv', sep='\t')
customers = pd.read_csv('customers.tsv', sep='\t')

# Set random seed for reproducibility
rng = np.random.default_rng(seed=42)
# Extract embeddings
def extract_embeddings(df):
    embeddings = df['FeatureVector'].apply(lambda x: np.fromstring(x.strip('[]'), sep=','))
    return np.vstack(embeddings.values)
c_embs = extract_embeddings(customers)
p_embs = extract_embeddings(products)
p_embs_normalized = p_embs / np.linalg.norm(p_embs, axis=1, keepdims=True)

# Initialize parameters summarizing purchase history for quick updates
# days_of_last_purchase = np.full((C, I), -np.inf) # for each customer-product pair
num_purchases = np.zeros(C, dtype=int) # for each customer
val_purchases = np.zeros(C) # for each customer

In [6]:
def sample_customer_preference(mean:np.ndarray, rng:np.random.Generator, cov: float=0.01) -> np.ndarray:
    '''Given the mean customer preference vector, sample a customer preference vector from a Gaussian distribution
    The covariance matrix is fixed as a multiple of the identity matrix for simplicity.
    The sampled output is clipped to be within [-1,1] for each dimension.
    Args:
        mean (np.ndarray): Mean customer preference vector
        rng (np.random.Generator): Random number generator
        cov (float): Covariance value to scale the identity matrix
    Returns:
        np.ndarray: Sampled customer preference vector'''

    sampled = rng.multivariate_normal(mean, cov * np.eye(len(mean)))
    return np.clip(sampled, -1, 1)

def get_recommendations(cid: int,  time_lapse:np.ndarray,
                        customers: pd.DataFrame=customers,
                        c_embs: np.ndarray=c_embs,
                        p_embs_normalized:np.ndarray=p_embs_normalized,  
                        K:int=K, rng:np.random.Generator=rng) -> list[int]:
    '''Given a customer embedding and product embeddings, return the top K recommended product indices
    Args:
        cid (int): Customer ID (starting from 0)
        time_lapse (np.ndarray): 1D array of days since last purchase for each product by the customer
        customers (pd.DataFrame): DataFrame containing customer data
        c_embs (np.ndarray): Customer feature matrix
        p_embs_normalized (np.ndarray): Normalized product embedding matrix
        K (int): Number of recommendations to return
        rng (np.random.Generator): Random number generator
    Returns:
        list[int]: List of top K recommended product indices
        scores (np.ndarray): Corresponding scores for the top K products'''
    assert 0 <= cid < customers.shape[0], "Customer ID out of range"
    c_emb = c_embs[cid]
    c_emb = sample_customer_preference(c_emb, rng)
    # Compute cosine similarity
    scores = p_embs_normalized @ c_emb / (np.linalg.norm(c_emb)+EPS)  
    # Decrease scores based on recency of last purchase
    scores = np.multiply(scores, 1- np.power(RHO, time_lapse))
    scores = np.clip(scores, 0, 1) # Ensure scores are within [0,1]
    top_k_indices = np.argsort(scores)[-K:][::-1]
    top_k_indices = top_k_indices.tolist()

    return top_k_indices, scores[top_k_indices]

In [7]:
def generate_purchase_history(customers: pd.DataFrame=customers,
                              c_embs: np.ndarray=c_embs,
                              products: pd.DataFrame=products,
                              start_date: dt.date=start_date,
                              T: int=T,
                              AT_RISK_DAYS: int=AT_RISK_DAYS,
                              purchase_rate: float=0.1,
                              p_exit: float=0.3,
                              MAX_ORDERS: int=80,
                              rng: np.random.Generator=rng) -> list:
    '''Generate a simulated purchase history for customers over a time period T
    Args:
        customers (pd.DataFrame): DataFrame containing customer data
        c_embs (np.ndarray): Customer feature matrix
        products (pd.DataFrame): DataFrame containing product data
        start_date (dt.date): Start date for the simulation
        T (int): Total number of days to simulate
        AT_RISK_DAYS (int): Number of days to consider a regular customer at-risk
        purchase_rate (float): Purchase rate for the customer
        p_exit (float): Probability of ending shopping session
        MAX_ORDERS (int): Maximum number of orders to sample from Poisson point process
        rng (np.random.Generator): Random number generator
    Returns:
        list: Simulated purchase history as a list of (tid, cid, day, [(item_id, quantity), ...])'''
    # Store the purchase history as a list of (tid, cid, day, [(item_id, quantity), ...])
    purchase_history = []
    tid = 0
    # Simulate purchase history for each customer
    for cid, s, c_emb in zip(customers.CID, customers.Segment, c_embs[customers.index]):
        if s == 'N': # no need to simulate purchases
            continue
        # Determine the last day to consider purchases for this customer
        STOP_DAY = T - AT_RISK_DAYS - 1 if s == 'A' else T-1
        # Add some noise to the rate
        rate = np.clip(purchase_rate + rng.normal(0, 0.01), 0.02, None) 
        
        # Simulate purchase times using Poisson process
        purchase_times = rng.exponential(1/rate, MAX_ORDERS).cumsum() # inter-arrival times
        purchase_days = np.ceil(purchase_times).astype(int)
        purchase_days = purchase_days[purchase_days <= STOP_DAY] # truncate

        # Make sure that for non A customers, there is at least one purchase in the last AT_RISK_DAYS
        if s in ['L','R','H'] and  purchase_days[-1] < STOP_DAY - AT_RISK_DAYS:
            # print(f'Adding additional purchase for customer {cid} with segment {s}')
            purchase_days = np.append(purchase_days, rng.integers(T-AT_RISK_DAYS+1, T-1))

        assert len(purchase_days) > 0, f"No purchases generated for customer {cid} in segment {s}"
        assert all(purchase_days[i] <= purchase_days[i+1] for i in range(len(purchase_days)-1)), "Purchase days not sorted"
        assert all(0 <= d <= STOP_DAY for d in purchase_days), "Purchase days out of bounds"
        # Drop duplicates by keeping only one order per day
        purchase_days = [int(d) for d in sorted(set(purchase_days))]
        final_day = purchase_days[-1] # force purchase on last day sampled to meet constraint

        # Simulate purchases for each day
        days_of_last_purchase = np.full(I, -np.inf)

        for day in purchase_days:
            # Get recommendations
            time_lapse = day - days_of_last_purchase
            recs_id, recs_scores = get_recommendations(cid, customers=customers,time_lapse=time_lapse, c_embs=c_embs, rng=rng)

            # Simulate purchases 
            cart = [] # initialize empty cart
            for ii, (r_id, r_score) in enumerate(zip(recs_id, recs_scores)):
                if rng.binomial(1, p_exit): # end shopping session with probability p_exit
                    break
                add_to_cart = rng.binomial(1, r_score) # decide whether to add to cart
                if add_to_cart:
                    q = int(np.clip(rng.poisson(0.5),1, 4)) # decide quantity
                    cart.append((r_id, q))
            
            # on the final day, ensure at least one purchase
            if day==final_day and len(cart)==0:
                r_id = recs_id[0] # pick the top recommendation
                q = int(np.clip(rng.poisson(0.5),1, 4)) # decide quantity
                cart.append((r_id, q))
            
            # Record the purchase if any items were bought
            if len(cart) > 0:
                purchase_history.append((tid, cid, start_date + dt.timedelta(days=day), cart))
                tid += 1
                # Update tracking parameters
                items_bought = [c[0] for c in cart]
                days_of_last_purchase[items_bought] = day
    return purchase_history


In [ ]:
# # generate purchase history
# purchase_history = generate_purchase_history(customers=customers)
# df = pd.DataFrame(purchase_history, columns=['TID','CID', 'Date', 'Order'])
# # compute the size and profit of orders
# profits = products.Price - products.Cost
# compute_order_size = lambda order: sum(quantity for item_id, quantity in order)
# compute_order_profit = lambda order: sum(profits.loc[item_id] * quantity for item_id, quantity in order)
# df['OrderProfit'] = df['Order'].apply(compute_order_profit)
# df['OrderSize'] = df['Order'].apply(compute_order_size)
# df.sort_values(by=['Date', 'CID'], inplace=True)

In [ ]:
# df.to_csv('transactions.tsv', sep='\t', index=False)

## Check correctness
Note: we skip the verification because the item prices were changed and we no longer need the purchase history.

In [ ]:
# # for each customer, compute their average profit over time T
# avg_profit = df.groupby('CID')['OrderProfit'].sum().reset_index().rename(columns={'OrderProfit':'AvgProfit'})
# avg_profit['AvgProfit'] = avg_profit['AvgProfit'] / T
# # add to customer dataframe
# customers = customers.merge(avg_profit, on='CID', how='left')

In [12]:
# # check if average profit is consistent with segment
# assert all(customers.loc[customers['Segment']=='H', 'AvgProfit'] >= AVG_VAL_H), "High-value customers have AvgProfit below threshold"
# customers[(customers['Segment'] != 'A') & (customers['Segment'] != 'N')].sort_values('AvgProfit', ascending=False)

In [11]:
# # check if all at-risk customers have recent purchase requirement at least AT_RISK_DAYS ago
# print('Days since last purchase:')
# for cid in customers[customers.Segment=='A'].CID:
#     delta = (end_date - df[df.CID==cid].iloc[-1].Date).days
#     print(f'Customer {cid}: {delta}')
#     assert  delta> AT_RISK_DAYS


In [13]:
# # check if all non-at-risk, non-new customers have recent purchase requirement at most AT_RISK_DAYS ago
# print('Days since last purchase:')
# for cid in customers[customers.Segment.isin(['L','R','H'])].CID:
#     delta = (end_date - df[df.CID==cid].iloc[-1].Date).days
#     print(f'Customer {cid}: {delta}')
#     assert  delta<= AT_RISK_DAYS


## New for Project 2: Patching data distribution

It was observed that the original feature vectors result in undesirable preference distributions:
1. Most customers like most items, so random baseline performs well
2. Sophisticated RS does not beat the baseline of recommending the most expensive items

To fix this issue, we make some changes to include prices as part of the feature and decrease the price of the most expensive items. Specifically, we do the following:
0. Change the price and cost of items to remove outliers.
1. Multiply each entry in each customer preference vector by iid beta random variables reasonably concentrated around 1. After that, append 1 to each customer preference vector to account for price.
2. Append a price feature to each item vector, calculated as a sigmoid function which is scaled, shifted and flipped, so that the range is $[-1,1]$ and higher prices give values closer to $-1$.


In particular, the feature dimension has increased from 5 to 6. We do not generate past purchase history anymore, as dynamic customer segmentation was not used in project 1 and RS in project 2 allows for cold start.


In [5]:
# 1. Slightly scale down original feature part to emphasize price feature
rng = np.random.default_rng(42)
customer_pref_mult = rng.beta(a=6, b=1.2, size=5*40).reshape(40, 5)
customer_preferences = c_embs * customer_pref_mult
customer_preferences = np.hstack((customer_preferences, np.ones((40,1))))
print(f"Customer preference vectors shape: {customer_preferences.shape}")

Customer preference vectors shape: (40, 6)


In [6]:
# 2. Compute item price features using a sigmoid function
def sigmoid(x: float, scale: float=0.02, center: float=180) -> float:
    """Sigmoid function."""
    return 1 / (1 + np.exp(scale * (x - center)))
    
def compute_item_price_features(price: float) -> float:
    """Calculate item price features."""
    return 2*sigmoid(price)-1
item_price_features = np.vectorize(compute_item_price_features)(products.Price.to_numpy())
item_features =np.hstack((p_embs, item_price_features.reshape(-1,1)))
print(f"Item feature vectors shape: {item_features.shape}")


Item feature vectors shape: (50, 6)


## Database Population

Now that we have generated all the mock data, let's populate the SQLite database with this data.

In [7]:
import sys
sys.path.append('..')
from models.database import initialize_database, close_database
from models.models import Customer
from models.models import Item
from models.models import Transaction
import ast
import os
import json
import numpy as np

# Remove existing database to make this idempotent
db_path = "retail_project.db"
if os.path.exists(db_path):
    os.remove(db_path)
    print("Removed existing database for fresh start")

# Initialize Peewee database and create tables
initialize_database()
print("=== Populating Database with Mock Data using Peewee ORM ===\n")


# 1. Insert customers
print("1. Inserting customers...")
for i, customer in customers.iterrows():
    # Get customer preference vector
    feature_vector = list(customer_preferences[i]) # ast.literal_eval(customer['FeatureVector'])
    
    # For now, use default location - could be enhanced later
    location = (40.7128 + np.random.normal(0, 0.01), -74.0060 + np.random.normal(0, 0.01))

    Customer.create(
        cid=customer['CID'],
        name=customer['Name'],
        age=customer['Age'],
        gender=customer['Gender'],
        location_x=location[0],
        location_y=location[1],
        job_type=customer['JobType'],
        income=customer['Income'],
        segment=customer['Segment'],
        family_status=customer['FamilyStatus'],
        feature_vector=feature_vector,
        satisfaction=0.5,  # Default satisfaction
        visit_probability=0.5,  # Default visit probability
        session_probability=0.3,  # Default session probability
        trigger_keywords=[]  # Default empty keywords
    )
print(f"✓ Inserted {len(customers)} customers")

# 2. Insert products
print("\n2. Inserting products...")
for i, product in products.iterrows():
    # Get product feature vector
    feature_vector = list(item_features[i]) # ast.literal_eval(product['FeatureVector'])
    Item.create(
        pid=product['PID'],
        name=product['Name'],
        description=product['Description'],
        price=product['Price'],
        discounted=False,  # Default value
        cost=product['Cost'],
        stock_level=50,  # Default stock level
        feature_vector=feature_vector
    )
print(f"✓ Inserted {len(products)} products")

# # 3. Insert transactions 
# print("\n3. Inserting transactions...")

# for _, transaction in df.iterrows():
#     # Parse the order data
#     order_items = []
#     for item_id, quantity in transaction['Order']:
#         price = products.loc[products['PID'] == item_id, 'Price'].iloc[0]
#         order_items.append((item_id, quantity, price))
    
#     # Convert date to simulation day (days since start_date)
#     transaction_date = transaction['Date']
    
#     # Create Transaction using Peewee
#     Transaction.create(
#         tid=transaction['TID'],
#         cid=transaction['CID'],
#         date=transaction_date,
#         order_data=order_items,
#         delivery_time_window_start=9,  # Default delivery window start
#         delivery_time_window_end=17   # Default delivery window end
#     )

# print(f"✓ Inserted {len(df)} transactions")

print("\n=== Database Population Complete ===")

Removed existing database for fresh start
=== Populating Database with Mock Data using Peewee ORM ===

1. Inserting customers...
✓ Inserted 40 customers

2. Inserting products...
✓ Inserted 50 products

=== Database Population Complete ===


### Verification

Let's verify that the data was inserted correctly into the database.

In [8]:
# Verify database contents using Peewee models
print("=== Database Verification ===\n")

# Check customers table
customer_count = Customer.select().count()
print(f"Customers in database: {customer_count}")

# Check products table
product_count = Item.select().count()
print(f"Products in database: {product_count}")

# Check transactions table
transaction_count = Transaction.select().count()
print(f"Transactions in database: {transaction_count}")

# Sample some data
print(f"\n--- Sample Customer ---")
try:
    sample_customer = Customer.get(Customer.cid == 0)
    print(f"Name: {sample_customer.name}")
    print(f"Age: {sample_customer.age}")
    print(f"Segment: {sample_customer.segment}")
except Customer.DoesNotExist:
    print("Customer 0 not found")

print(f"\n--- Sample Product ---")
try:
    sample_product = Item.get(Item.pid == 0)
    print(f"Name: {sample_product.name}")
    print(f"Price: ${sample_product.price}")
except Item.DoesNotExist:
    print("Product 0 not found")

print(f"\n--- Sample Transaction ---")
sample_transactions = Transaction.select().where(Transaction.cid == 0)
if sample_transactions.exists():
    transaction_count = sample_transactions.count()
    first_transaction = sample_transactions.get()
    print(f"Customer 0 has {transaction_count} transaction(s)")
    print(f"First order: {first_transaction.order_data}")
else:
    print("No transactions found for customer 0")

print(f"\n✓ Database verification complete!")
print(f"Database file location: data/demo_retail.db")
print(f"The notebook is now idempotent - run it multiple times safely!")

# Close database connection
close_database()

=== Database Verification ===

Customers in database: 40
Products in database: 50
Transactions in database: 0

--- Sample Customer ---
Name: Liam Chen
Age: 45
Segment: H

--- Sample Product ---
Name: Aethelred Luxury Chronograph
Price: $499.0

--- Sample Transaction ---
No transactions found for customer 0

✓ Database verification complete!
Database file location: data/demo_retail.db
The notebook is now idempotent - run it multiple times safely!
